<a href="https://colab.research.google.com/github/trefftzc/cis677/blob/main/voronoi_cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# An implementation in CUDA of an approximate Voronoi Diagram

In [ ]:
%%writefile voronoi_cuda_64_64.cu
#include <cstdio>
#include <cuda_runtime.h>
// #include <cmath>
// The size of the grid is 64 x 64
const int N = 64;

// An auxiliary function to calculate the euclidean distance between 2 points
__device__ int distance(int x1,int y1,int x2,int y2) {
  int aux_x = x2 - x1;
  int aux_y = y2 - y1;
  aux_x = aux_x * aux_x;
  aux_y = aux_y * aux_y;
  int sum = aux_x + aux_y;
  return (int) sqrtf(sum);
  }

// The kernel that calculates the approximate Voronoi Diagram
__global__ void calcVoronoi(int* data, size_t pitch, int width, int height,
int nSeeds,int2 *seeds)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < width && y < height) {
        int* row = reinterpret_cast<int*>(reinterpret_cast<char*>(data) + y * pitch);
	// Calculate the id of the closest seed
	int closestSeed = -1;
	int closestDistance = N*N;
	for(int i = 0;i < nSeeds;i++) {
		int seedx = seeds[i].x;
		int seedy = seeds[i].y;
		int dist = distance(seedx,seedy,x,y);
		if (dist < closestDistance) {
		   closestDistance = dist;
		   closestSeed = i;
		   }
		   }
        row[x] = closestSeed;
    }
}

int main()
{
    int width = N;
    int height = N;
    size_t pitch = 0;
    int* d_data = nullptr;
    // For a simple example, let's hardcode the number of seeds
    // and the values of the seeds
    int nSeeds = 4;
    int2 seeds[nSeeds];
    seeds[0] = make_int2(0,0);
    seeds[1] = make_int2(N,0);
    seeds[2] = make_int2(N,N);
    seeds[3] = make_int2(0,N);
    std::printf("%d %d\n",seeds[2].x,seeds[2].y);
    // Now allocate an array in the device memory for the seeds
    int2 *deviceSeeds = nullptr;
    cudaError_t err = cudaMalloc((void **)&deviceSeeds,nSeeds*sizeof(int2));
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMalloc failed: %s\n", cudaGetErrorString(err));
        return 1;

	}
// Copy the seeds from the host to device memory
    err = cudaMemcpy(deviceSeeds, seeds,nSeeds * sizeof(int2),
    cudaMemcpyHostToDevice);

    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMemcpy failed: %s\n", cudaGetErrorString(err));
        return 1;
    }
    // Allocate the 2d matrix that will contain the approximate voronoi diagram
    err = cudaMallocPitch(&d_data, &pitch, width * sizeof(int), height);
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMallocPitch failed: %s\n", cudaGetErrorString(err));
        return 1;

}


// Execute the kernel
    dim3 block(16, 16);
    dim3 grid((width + block.x - 1) / block.x, (height + block.y - 1) / block.y);
    calcVoronoi<<<grid, block>>>(d_data, pitch, width, height,nSeeds,deviceSeeds);

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaDeviceSynchronize failed: %s\n", cudaGetErrorString(err));
        cudaFree(d_data);
        return 1;
    }
// Copy the 2D array from the device to the host memory
    int* h_data = new int[width * height];
    err = cudaMemcpy2D(h_data, width * sizeof(int), d_data, pitch, width * sizeof(int), height, cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMemcpy2D failed: %s\n", cudaGetErrorString(err));
        delete[] h_data;
        cudaFree(d_data);
        return 1;
    }
// Print sample data
//    std::printf("sample [0][0] = %d, [1023][1023] = %d\n", h_data[0], h_data[1023 * width + 1023]);
    for(int i = 0;i < N;i++) {
      for(int j = 0;j < N;j++) {
        std::printf("%d", h_data[i*N+j]);
	}
	std::printf("\n");
	}
    delete[] h_data;
    cudaFree(d_data);
    cudaFree(deviceSeeds);
    return 0;
}

Writing voronoi_cuda_64_64.cu


In [ ]:
!nvcc  voronoi_cuda_64_64.cu -o voronoi_cuda_64_64


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./voronoi_cuda_64_64

64 64
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000000000000000001111111111111111111111111111111
0000000000000000000

In [ ]:
%%writefile voronoi_cuda_2048_2048.cu
#include <cstdio>
#include <cuda_runtime.h>
// #include <cmath>
// The size of the grid is 2048 x 2048
const int N = 2048;

// An auxiliary function to calculate the euclidean distance between 2 points
__device__ int distance(int x1,int y1,int x2,int y2) {
  int aux_x = x2 - x1;
  int aux_y = y2 - y1;
  aux_x = aux_x * aux_x;
  aux_y = aux_y * aux_y;
  int sum = aux_x + aux_y;
  return (int) sqrtf(sum);
  }

// The kernel that calculates the approximate Voronoi Diagram
__global__ void calcVoronoi(int* data, size_t pitch, int width, int height,
int nSeeds,int2 *seeds)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < width && y < height) {
        int* row = reinterpret_cast<int*>(reinterpret_cast<char*>(data) + y * pitch);
	// Calculate the id of the closest seed
	int closestSeed = -1;
	int closestDistance = N*N;
	for(int i = 0;i < nSeeds;i++) {
		int seedx = seeds[i].x;
		int seedy = seeds[i].y;
		int dist = distance(seedx,seedy,x,y);
		if (dist < closestDistance) {
		   closestDistance = dist;
		   closestSeed = i;
		   }
		   }
        row[x] = closestSeed;
    }
}

int main()
{
    int width = N;
    int height = N;
    size_t pitch = 0;
    int* d_data = nullptr;
    // For a simple example, let's hardcode the number of seeds
    // and the values of the seeds
    int nSeeds = 4;
    int2 seeds[nSeeds];
    seeds[0] = make_int2(0,0);
    seeds[1] = make_int2(N,0);
    seeds[2] = make_int2(N,N);
    seeds[3] = make_int2(0,N);
    // std::printf("%d %d\n",seeds[2].x,seeds[2].y);
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    // Now allocate an array in the device memory for the seeds
    int2 *deviceSeeds = nullptr;
    cudaError_t err = cudaMalloc((void **)&deviceSeeds,nSeeds*sizeof(int2));
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMalloc failed: %s\n", cudaGetErrorString(err));
        return 1;

	}
// Copy the seeds from the host to device memory
    err = cudaMemcpy(deviceSeeds, seeds,nSeeds * sizeof(int2),
    cudaMemcpyHostToDevice);

    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMemcpy failed: %s\n", cudaGetErrorString(err));
        return 1;
    }
    // Allocate the 2d matrix that will contain the approximate voronoi diagram
    err = cudaMallocPitch(&d_data, &pitch, width * sizeof(int), height);
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMallocPitch failed: %s\n", cudaGetErrorString(err));
        return 1;

}


// Execute the kernel
    dim3 block(16, 16);
    dim3 grid((width + block.x - 1) / block.x, (height + block.y - 1) / block.y);
    cudaEventRecord(start, 0);
    calcVoronoi<<<grid, block>>>(d_data, pitch, width, height,nSeeds,deviceSeeds);

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaDeviceSynchronize failed: %s\n", cudaGetErrorString(err));
        cudaFree(d_data);
        return 1;
    }
    cudaEventRecord(stop, 0);
// Copy the 2D array from the device to the host memory
    int* h_data = new int[width * height];
    err = cudaMemcpy2D(h_data, width * sizeof(int), d_data, pitch, width * sizeof(int), height, cudaMemcpyDeviceToHost);
    if (err != cudaSuccess) {
        std::fprintf(stderr, "cudaMemcpy2D failed: %s\n", cudaGetErrorString(err));
        delete[] h_data;
        cudaFree(d_data);
        return 1;
    }
// Print sample data
//    std::printf("sample [0][0] = %d, [1023][1023] = %d\n", h_data[0], h_data[1023 * width + 1023]);
/*
    for(int i = 0;i < N;i++) {
      for(int j = 0;j < N;j++) {
        std::printf("%d", h_data[i*N+j]);
	}
	std::printf("\n");
	}
*/
    // Print the time taken by the execution of the kernel
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    std::printf("Kernel Execution Time: %f milliseconds.\n",milliseconds);
    delete[] h_data;
    cudaFree(d_data);
    cudaFree(deviceSeeds);
    return 0;
}

Writing voronoi_cuda_2048_2048.cu


In [ ]:
!nvcc  voronoi_cuda_2048_2048.cu -o voronoi_cuda_2048_2048

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./voronoi_cuda_2048_2048

Kernel Execution Time: 23.105536 milliseconds.


In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
